# 23 — Errori R2L e U2R per attacco specifico (T39)

Leggiamo tutte le predizioni già salvate per R2L e U2R. Non cambiamo etichette,
modelli o parametri. Il modello predice la **famiglia**, non il nome specifico
dell'attacco: il nome serve soltanto a raggruppare i risultati della diagnosi.

Per ogni attacco mostriamo quanti esempi esistevano nel training, quanti sono nel
test e dove finiscono gli errori. I gruppi piccoli vanno letti come conteggi, non
come stime precise della prestazione generale.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, f1_score, accuracy_score

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import COLUMN_NAMES
from src.targets import AUTHOR_ATTACK_TO_FAMILY, FAMILY_ORDER
OUTPUT = ROOT / 'artifacts/rare_attack_subtypes'
OUTPUT.mkdir(parents=True, exist_ok=True)
protocol_path = ROOT / 'configs/official_test_v1.json'
protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
for name, digest in protocol['input_sha256'].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest
test = pd.read_csv(ROOT / 'data/raw/KDDTest+.csv', header=None, names=COLUMN_NAMES)
assert test['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()
expected_truth = test['attack_label'].map(AUTHOR_ATTACK_TO_FAMILY).to_numpy(dtype=str)
models = {
    'decision_tree': ('Decision Tree', 'decision_tree_official_test'),
    'random_forest_balanced': ('Random Forest', 'random_forest_balanced_official_test'),
    'gaussian_naive_bayes': ('Gaussian Naive Bayes', 'gaussian_naive_bayes_official_test'),
    'svm_rbf': ('SVM RBF', 'svm_rbf_official_test'),
    'neural_network': ('Rete neurale', 'neural_network_official_test'),
    'kmeans': ('K-Means', 'kmeans_official_test'),
}
train = pd.read_csv(ROOT / 'data/raw/KDDTrain+.csv', header=None, names=COLUMN_NAMES)
assert train['attack_label'].isin(AUTHOR_ATTACK_TO_FAMILY).all()

## 1. Verificare le predizioni salvate

Stesso protocollo, parametri, input, target e indici dei sei esperimenti ufficiali. Ricalcoliamo le confusion matrix per verificare la coerenza degli artefatti. Nessun fit o nuova predizione.

In [ ]:
prediction_frames = []
overall_rows = []
class_rows = []
matrices = {}
saved_results = {}
for model_id, (label, artifact) in models.items():
    folder = ROOT / 'artifacts' / artifact
    result = json.loads((folder / 'results.json').read_text(encoding='utf-8'))
    assert result['protocol_sha256'] == protocol_hash
    assert result['protocol_id'] == protocol['protocol_id']
    assert result['input_sha256'] == protocol['input_sha256']
    assert result['label_profile'] == protocol['label_profile']
    assert result['family_order'] == list(FAMILY_ORDER)
    assert result['train_rows'] == 125973 and result['test_rows'] == len(expected_truth)
    assert result['classifier_parameters'] == protocol['models'][model_id]['parameters']
    assert hashlib.sha256((folder / 'predictions.npz').read_bytes()).hexdigest() == result['predictions_sha256']
    with np.load(folder / 'predictions.npz', allow_pickle=False) as saved:
        truth, predicted = saved['truth'], saved['predicted']
        np.testing.assert_array_equal(saved['test_indices'], np.arange(len(expected_truth)))
    np.testing.assert_array_equal(truth, expected_truth)
    assert set(predicted) <= set(FAMILY_ORDER)
    counts = confusion_matrix(truth, predicted, labels=FAMILY_ORDER)
    normalized = confusion_matrix(truth, predicted, labels=FAMILY_ORDER, normalize='true')
    np.testing.assert_array_equal(counts, result['confusion_counts'])
    np.testing.assert_allclose(normalized, result['confusion_normalized'])
    precision, recall, f1, support = precision_recall_fscore_support(truth, predicted, labels=FAMILY_ORDER, zero_division=0)
    scores = {
        'macro_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='macro', zero_division=0),
        'weighted_f1': f1_score(truth, predicted, labels=FAMILY_ORDER, average='weighted', zero_division=0),
        'accuracy': accuracy_score(truth, predicted),
    }
    for metric, value in scores.items():
        np.testing.assert_allclose(value, result['metrics'][metric])
    overall_rows.append({'model_id': model_id, 'model': label, **scores, 'fit_seconds': result['fit_seconds']})
    for i, family in enumerate(FAMILY_ORDER):
        class_rows.append({'model_id': model_id, 'model': label, 'family': family,
                           'precision': precision[i], 'recall': recall[i], 'f1': f1[i],
                           'support': int(support[i]), 'true_positives': int(counts[i,i]),
                           'false_positives': int(counts[:,i].sum()-counts[i,i]),
                           'error_rate': 1-recall[i]})
    prediction_frames.append(pd.DataFrame({
        'model_id': model_id, 'model': label, 'test_index': np.arange(len(truth)),
        'attack': test['attack_label'].to_numpy(), 'family': truth, 'predicted': predicted,
    }))
    matrices[model_id] = normalized
    saved_results[model_id] = result
overall = pd.DataFrame(overall_rows)
per_class = pd.DataFrame(class_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)

## 2. Contare i supporti nel training

Una quantità pari a zero significa che quel tipo specifico non era presente nel
training. Non significa che la famiglia R2L/U2R fosse sconosciuta al modello.

In [ ]:
train_counts = train['attack_label'].value_counts()
predictions['train_support'] = predictions['attack'].map(train_counts).fillna(0).astype(int)

## 3. Separare i tre esiti

Ogni riga rara ricade in una sola categoria: famiglia corretta, classificata Normal,
oppure assegnata a un'altra famiglia di attacco. La somma deve coincidere con il
supporto del gruppo. Nessuna riga Normal entra in questo approfondimento.

In [ ]:
rare = predictions[predictions['family'].isin(['R2L','U2R'])].copy()
rare['family_correct'] = rare['family'].eq(rare['predicted'])
rare['predicted_normal'] = rare['predicted'].eq('Normal')
rare['wrong_attack_family'] = ~rare['family_correct'] & ~rare['predicted_normal']
subtype_summary = rare.groupby(['model','family','attack','train_support'], sort=False).agg(
    support=('family','size'), family_correct=('family_correct','sum'),
    predicted_normal=('predicted_normal','sum'), wrong_attack_family=('wrong_attack_family','sum'),
).reset_index()
assert (subtype_summary['support'] == subtype_summary[['family_correct','predicted_normal','wrong_attack_family']].sum(axis=1)).all()
subtype_summary['present_in_train'] = subtype_summary['train_support'].gt(0)
subtype_summary['family_recall'] = subtype_summary['family_correct']/subtype_summary['support']
subtype_summary['normal_error_rate'] = subtype_summary['predicted_normal']/subtype_summary['support']
subtype_summary['attack_detection_recall'] = 1-subtype_summary['normal_error_rate']

In [ ]:
display(subtype_summary.sort_values(['family','support','attack','model'],ascending=[True,False,True,True]).round(4))
destination_counts = pd.crosstab(
    [rare['model'],rare['family'],rare['attack']],rare['predicted'],
).reindex(columns=FAMILY_ORDER,fill_value=0)
display(destination_counts)

## 4. Recall della famiglia per tipo di attacco

Ogni riga mostra il nome specifico, il numero di casi nel training e nel test.
Le celle riportano la percentuale assegnata alla famiglia corretta. Sono inclusi
tutti gli attacchi R2L/U2R presenti nel test, anche quelli con uno o pochi esempi.

In [ ]:
model_labels = [label for label, _ in models.values()]
short_labels = ['DT','RF','GaussianNB','SVM','MLP','K-Means']
for family in ['R2L','U2R']:
    subset = subtype_summary[subtype_summary['family'].eq(family)]
    support = subset[['attack','train_support','support']].drop_duplicates().sort_values(['support','attack'],ascending=[False,True])
    values = subset.pivot(index='attack',columns='model',values='family_recall').reindex(index=support['attack'],columns=model_labels)
    fig,axis = plt.subplots(figsize=(10,max(3.4,len(support)*.43+1.4)),layout='constrained')
    im=axis.imshow(values.to_numpy(),vmin=0,vmax=1,cmap='Blues',aspect='auto')
    for i in range(len(support)):
        for j in range(len(model_labels)):
            value=values.iloc[i,j]
            axis.text(j,i,f'{value*100:.1f}',ha='center',va='center',fontsize=9,color='white' if value>.55 else '#143755')
    row_labels=[f'{r.attack}  (train {r.train_support}; test {r.support})' for r in support.itertuples()]
    axis.set(xticks=range(6),xticklabels=short_labels,yticks=range(len(support)),yticklabels=row_labels,
             title=f'{family} — famiglia corretta (%) per attacco specifico')
    fig.colorbar(im,ax=axis,label='Recall della famiglia')
    fig.savefig(OUTPUT/f'{family.lower()}_subtype_recall.png',dpi=160,bbox_inches='tight')
    plt.show()

## 5. Peso degli errori e riepilogo

I conteggi verso Normal e verso altre famiglie separano mancato rilevamento ed
errore di classificazione. Ordinare gli attacchi per numero di errori aiuta a
individuare dove si concentra il problema, senza confondere una percentuale alta
su pochi casi con un contributo elevato al totale.

In [ ]:
subtype_summary['family_errors'] = subtype_summary['support']-subtype_summary['family_correct']
family_totals = subtype_summary.groupby(['model','family'],sort=False)[
    ['support','family_correct','predicted_normal','wrong_attack_family','family_errors']
].sum().reset_index()
for label in model_labels:
    total=family_totals[family_totals['model'].eq(label)].set_index('family')
    assert total.loc['R2L','support']==2754 and total.loc['U2R','support']==200
display(family_totals)
display(subtype_summary.sort_values(['model','family_errors'],ascending=[True,False]).round(4))

In [ ]:
subtype_summary.to_csv(OUTPUT/'subtype_metrics.csv',index=False)
family_totals.to_csv(OUTPUT/'family_totals.csv',index=False)
destination_counts.to_csv(OUTPUT/'prediction_destinations.csv')
(OUTPUT/'verification.json').write_text(json.dumps({
    'protocol_sha256':protocol_hash,'model_count':len(models),'model_fits':0,'new_predictions':0,
    'rare_rows_per_model':{'R2L':2754,'U2R':200},
    'input_sha256':protocol['input_sha256'],
},indent=2)+'\n',encoding='utf-8')

## Limiti

Analisi descrittiva del test già consultato: non permette di attribuire causalmente
gli errori alla novità dell'attacco o allo sbilanciamento. Ricordiamo la convenzione
autore: httptunnel appartiene a U2R e worm a R2L. Le famiglie sono note al training;
alcuni nomi specifici no. Nessuna nuova configurazione è stata provata.

## Risultato della diagnosi

guess_passwd era presente nel training (53 casi), ma RF e MLP classificano tutti
i suoi 1.231 casi del test come Normal. Per U2R, httptunnel rappresenta 133/200 casi
ed era assente dal training: MLP ne classifica 130 come Normal e 1 come U2R.
Per K-Means, 113/133 sono rilevati come attacchi ma solo 1 come U2R: rilevamento e
classificazione della famiglia misurano due aspetti diversi.
Tutte le righe, anche quelle con supporto minimo, sono riportate senza escludere
gli esiti sfavorevoli. Sintesi e limiti in docs/ATTACK_DIAGNOSIS.md.
